# Task 2 · T4 16GB / Qwen2.5-7B-Instruct-AWQ 兼容性试跑

在 Colab「运行时 → 更改运行时类型」选择 T4 GPU，再依次运行。本 notebook 不创建 GCP 资源、不兑换券。Colab 的 GPU 可用性、会话期限和收费由当前账号决定。

当前仅对齐了 T4 和 AWQ 模型；vLLM 等参数尚待任务三、四同步。所有输出标记为 **candidate trial**，不能作为全组正式对照数据。每次测量预热后清空前缀缓存，profiling 独立执行。原版参考使用 IBM 官方 direct_llm prompt，不是完整 MCP Agent。


In [ ]:
import subprocess
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True))
rows = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True).strip().splitlines()
assert len(rows) == 1 and rows[0].split(',')[0].strip() == 'Tesla T4', 'This trial requires exactly one Tesla T4; do not silently switch GPU.'


In [ ]:
from pathlib import Path
repo = Path('/content/assetopsbench-vllm-baseline')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/KevinHu-Create/assetopsbench-vllm-baseline.git', str(repo)], check=True)
# Pin the tested implementation. Later documentation-only commits do not change this trial.
subprocess.run(['git', 'checkout', '483b435f1d520b4894eb16b3fc9886654d7c9ad0'], cwd=repo, check=True)
subprocess.run(['bash', 'scripts/bootstrap_gpu.sh'], cwd=repo, check=True)


In [ ]:
py = str(repo / '.venv/bin/python')
subprocess.run([py, '-m', 'task2.team', 'snapshot', '--environment', 'outputs/environment.json', '--out', 'outputs/t4-candidate.json'], cwd=repo, check=True)
print((repo / 'outputs/t4-candidate.json').read_text())


In [ ]:
subprocess.run([py, 'scripts/run_gpu.py', '--trial', '--team-profile', 'outputs/t4-candidate.json', '--out', 'outputs/t4-trial'], cwd=repo, check=True)


In [ ]:
import json
for c in (1, 4, 8):
    d = json.loads((repo / f'outputs/t4-trial/baseline-c{c}/run_summary.json').read_text())
    print(c, d['measurement_kind'], json.dumps(d['aggregate'], ensure_ascii=False, indent=2))


## 下载证据与结束会话

以下 ZIP 包含模型配置、服务日志、请求日志、结果表及短 CUDA trace；不含券码或 GCP 账号信息。若试跑失败，请先查看 `outputs/t4-trial/server/launcher.log`、`server.log`，不要把失败输出写成正式性能结果。

下载完后在 Colab「运行时 → 断开连接并删除运行时」释放 GPU。该 notebook 没有后台创建的 GCP VM。


In [ ]:
import shutil
archive = shutil.make_archive('/content/task2-t4-trial', 'zip', repo / 'outputs')
from google.colab import files
files.download(archive)
